In [ ]:
import re
import pandas as pd
import emoji
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Caminho arquivo exportado do wpp
caminho_arquivo = r'C:\Users\5515998\Downloads\_chat.txt'

In [ ]:
def converter_emojis(lista_emojis):
    return [emoji.demojize(e) for e in lista_emojis]

# regex mensagens gerais
padrao_mensagem = r'\[(\d{2}/\d{2}/\d{4}) (\d{2}:\d{2}:\d{2})\] (.*?): (.*)'
padrao_mensagem_erro = r'\[(\d{2}/\d{2}/\d{4}) (\d{2}:\d{2}:\d{2})\] (.*?):'
padrao_data_hora = r'\[\d{2}/\d{2}/\d{4} \d{2}:\d{2}:\d{2}\]'

dados = []

mensagem_atual = ""
data = ""
hora = ""
nome = ""

with open(caminho_arquivo, 'r', encoding='utf-8') as arquivo:
    for linha in arquivo: 
        # Tirando os caractere bugado
        linha = linha.strip('\u200e')
        linha = linha.strip()
        
        # correspondendo com o regex
        correspondencia = re.match(padrao_mensagem, linha)
        correspondencia_data_hora = re.match(padrao_data_hora, linha)
        
        if correspondencia_data_hora: # a msg costuma começar com data e hora
            # se msg atual nao for false, quer dizer q é referente a msg da linha anterior
            if mensagem_atual:
                dados.append({
                    "Data": data,
                    "Hora": hora,
                    "Nome": nome,
                    "Mensagem": mensagem_atual.strip(),  # Remove espaços em branco desnecessários
                })
                # resetando a msg atual
                mensagem_atual = False
            
            # Tenta pegar os dados da msg
            try:
                data, hora, nome, mensagem = correspondencia.groups()
            except:
                correspondencia_erro = re.match(padrao_mensagem_erro, linha)
                data, hora, nome = correspondencia_erro.groups()
                mensagem = ''
                # print(linha)
                # print(nome, data, hora)
            mensagem_atual = mensagem  # Inicializa a mensagem atual com a nova mensagem
        else:
            # Se for uma mensagem com quebra de linha, coloca na mensagem "atual"
            mensagem_atual += linha  # somando a string

    # Adiciona a última mensagem após o loop
    if mensagem_atual:
        dados.append({
            "Data": data,
            "Hora": hora,
            "Nome": nome,
            "Mensagem": mensagem_atual.strip(),
        })

df = pd.DataFrame(dados)

In [ ]:
df['Audio'] = False
df['Imagem'] = False
df['Gif'] = False
df['Figurinha'] = False
df['Emoji'] = False
df['Conteudo'] = 'Mensagem'
padrao_ligacao = r'iniciou uma ligação de vídeo'
padrao_figurinha = r'figurinha omitida'
padrao_audio = r'áudio ocultado'
padrao_imagem = r'imagem ocultada'
padrao_gif = r'GIF omitido'
padrao_emoji = r'[\U0001F600-\U0001F64F\U0001F300-\U0001F5FF\U0001F680-\U0001F6FF\U0001F1E0-\U0001F1FF]'

# preenchendo as colunas
df.loc[df['Mensagem'].str.contains(padrao_audio, na=False, regex=True), 'Audio'] = True
df.loc[df['Mensagem'].str.contains(padrao_imagem, na=False, regex=True), 'Imagem'] = True
df.loc[df['Mensagem'].str.contains(padrao_figurinha, na=False, regex=True), 'Figurinha'] = True
df.loc[df['Mensagem'].str.contains(padrao_gif, na=False, regex=True), 'Gif'] = True
df.loc[df['Mensagem'].str.contains(padrao_emoji, na=False, regex=True), 'Emoji'] = True

df.loc[df['Mensagem'].str.contains(padrao_audio, na=False, regex=True), 'Conteudo'] = 'Audio'
df.loc[df['Mensagem'].str.contains(padrao_imagem, na=False, regex=True), 'Conteudo'] = 'Imagem'
df.loc[df['Mensagem'].str.contains(padrao_figurinha, na=False, regex=True), 'Conteudo'] = 'Figurinha'
df.loc[df['Mensagem'].str.contains(padrao_gif, na=False, regex=True), 'Conteudo'] = 'GIF'
df.loc[df['Mensagem'].str.contains(padrao_emoji, na=False, regex=True), 'Conteudo'] = 'Com Emoji'

df['Emojis_Encontrados'] = df['Mensagem'].str.findall(padrao_emoji)
df['Emojis_Descritos'] = df['Emojis_Encontrados'].apply(converter_emojis)

In [ ]:
df['Data'] = pd.to_datetime(df['Data'], format="%d/%m/%Y")
df['MesAno'] = df['Data'].dt.to_period("M")

In [ ]:
df.query("Imagem == True").groupby(['MesAno', 'Nome']).agg(qtd_msg_mes=('Mensagem', 'count')).reset_index().sort_values(by=['MesAno', 'qtd_msg_mes'], ascending=False)

In [ ]:
df.groupby("Nome")['Data'].max().sort_values(ascending=True)

In [ ]:
df.query("Imagem == True").groupby(['MesAno', 'Nome'])['Mensagem'].count().reset_index().sort_values(by='Mensagem', ascending=False)

In [ ]:
df.head(2)

In [ ]:
df.query("Emojis_Encontrados.notna() | Emojis_Encontrados != '[]' ")['Emojis_Encontrados'].explode().reset_index().value_counts().reset_index().groupby('Emojis_Encontrados').agg(qtd_total=('count','sum')).reset_index().sort_values(by='qtd_total', ascending=False).head(20)

In [ ]:
df[df['Emoji'] == True]

In [ ]:
agrupado_data = df.query("Data >= '2024-01-01'").groupby("Data")['Mensagem'].count()

In [ ]:
agrupado_data

In [ ]:
d = agrupado_data.reset_index()
d

In [ ]:
d[d['Mensagem'] == d['Mensagem'].max()]

In [ ]:
agrupado_data_df = agrupado_data.reset_index()
agrupado_data_df.columns = ['Data', 'Contagem']

agrupado_data_df['Data'] = pd.to_datetime(agrupado_data_df['Data'])

agrupado_data_df = agrupado_data_df.sort_values('Data')
plt.figure(figsize=(15, 6))
sns.lineplot(data=agrupado_data_df)
media_contagem = agrupado_data_df['Contagem'].mean()
plt.axhline(media_contagem, color='red', linestyle='--', label='Média')
plt.title('QTD MSGS')
plt.xlabel('Data')
plt.ylabel('Contagem de Ocorrências')
plt.xticks(rotation=45)
plt.legend()

plt.show()

In [ ]:
df['Quantidade_de_k'] = df['Mensagem'].apply(lambda x: x.lower().count('k') + 1 if 'kk' in x.lower() else 0)

In [ ]:
df.query("Quantidade_de_k > 0").groupby("Nome").agg(Media_de_K_na_risada=('Quantidade_de_k', 'mean'),
                                                    Qtd_msg_com_kk=('Mensagem', 'count')).reset_index().sort_values(by='Media_de_K_na_risada', ascending=False)

In [ ]:
df.query("Data == '2024-12-18'").groupby('Nome')['Mensagem'].count().sort_values(ascending=False)